<a href="https://colab.research.google.com/github/Mizharrrrrhidi1818/EnsembleMethod-BinaryAdabost-GradientAdabost-MultiAdaboost/blob/main/EnsembleMethod_BinaryAdabost_GradientAdabost_MultiAdaboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0 Dataset Description

The dataset (**[train.csv](https://www.kaggle.com/datasets/rohitsahoo/sales-forecasting)**, 9,994 rows) resembles the well-known Superstore schema and covers U.S. office supply sales from 2015 to 2018. After grouping by Order ID, We worked with 4,922 unique transactions, each containing:

- Sales total per order (summed across line items)
- Categorical metadata: Category (e.g., Binders, Furniture, Technology), Segment (Consumer/Corporate/Home Office), Ship Mode, Region, and Order Date


In [ ]:
import pandas as pd, numpy as np
from pandas import DataFrame, Series
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import AdaBoostClassifier

In [ ]:
# 1. Load dataset anda data preprocessing
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rohitsahoo/sales-forecasting")

print("Path to dataset files:", path)

100%|██████████| 480k/480k [00:00<00:00, 641kB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/rohitsahoo/sales-forecasting/versions/2


In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
# This know our dataset after we have finished download

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import warnings
import tabulate
warnings.filterwarnings("ignore")

# Use the 'path' variable from the previous download cell to construct the correct file path
df = pd.read_csv(os.path.join(path, "train.csv"))

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   object 
 2   Order Date     9800 non-null   object 
 3   Ship Date      9800 non-null   object 
 4   Ship Mode      9800 non-null   object 
 5   Customer ID    9800 non-null   object 
 6   Customer Name  9800 non-null   object 
 7   Segment        9800 non-null   object 
 8   Country        9800 non-null   object 
 9   City           9800 non-null   object 
 10  State          9800 non-null   object 
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   object 
 13  Product ID     9800 non-null   object 
 14  Category       9800 non-null   object 
 15  Sub-Category   9800 non-null   object 
 16  Product Name   9800 non-null   object 
 17  Sales          9800 non-null   float64
dtypes: float

# 1 Exploration and preprocessing data

In [ ]:
# Check missing values
print(df.isnull().sum())

Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country           0
City              0
State             0
Postal Code      11
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
dtype: int64


In [ ]:
# Impute Missing Values(Postal code) with a known Vermont ZIP (e.g., Burlington, VT = 05401)
zip = df["Postal Code"].isnull() & \
       (df['City'] == 'Burlington') & \
       (df['State'] == 'Vermont')

df.loc[zip, 'Postal Code'] = '05401'

#Verify
print("Missing after imputation:", zip.isnull().sum())
print(df.isnull().sum())

Missing after imputation: 0
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
dtype: int64


We can change format date to ``Order Dat`` and ``Ship Date``column with using ``dayfirst`` (yy-mm-dd)

In [ ]:
# Convert Order Date and Ship Date to datetime with dayfirst=True
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst = True)
df["Ship Date"] = pd.to_datetime(df["Ship Date"], dayfirst = True)

# Drop 'Row ID' – not meaningful
df.drop(columns=["Row ID"], inplace=True)

In [ ]:
# Detect duplicate
df.duplicated().any()

np.True_

In [ ]:
df.duplicated().sum()

np.int64(1)

In [ ]:
duplicates = df[df.duplicated()]

duplicates

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
3406,US-2015-150119,2015-04-23,2015-04-27,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,Ohio,43229.0,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372


In [ ]:
# Removing duplicate
df = df.drop_duplicates()

df.duplicated().any()

np.False_

# 2 Feature Engineering

Feature engineering involves creating new features or modifying existing ones to improve the performance of machine learning models.

In [ ]:
# Aggregate by Order ID
order_df = df.groupby("Order ID").agg({
    "Sales": "sum",
    "Order Date": "first",
    "Ship Mode": "first",
    "Segment": "first",
    "Region": "first",
    "Category": lambda x: ', '.join(set(x)),
    "Sub-Category": lambda x: ', '.join(set(x))
}).reset_index()

# Discretize Sales into bins
order_df["Sales_bin"] = pd.cut(
    order_df["Sales"],
    bins=[0, 100, 300, 600, 10000],
    labels=["[0,100)", "[100,300)", "[300,600)", "[600+,10k]"],
    include_lowest=True
)

# Create Binary Classification Target: 0 = Low Sales, 1 = High Sales
order_df['Sales_binary'] = order_df['Sales_bin'].apply(
    lambda x: 1 if x in ["[300,600)", "[600+,10k]"] else 0
)

# Temporal features
order_df["Order_Weekday"] = order_df["Order Date"].dt.day_name()
order_df["Order_Month"] = order_df["Order Date"].dt.month_name().str[:3]

# Target for AdaBoost: {-1, 1}
order_df['Sales_binary_ab'] = order_df['Sales_bin'].apply(lambda x: 1 if x in ["[300,600)", "[600+,10k]"] else -1)
# Target for Gradient Boosting Classification: {0, 1}
order_df['Sales_binary_gb'] = order_df['Sales_bin'].apply(lambda x: 1 if x in ["[300,600)", "[600+,10k]"] else 0)

features = ["Order_Weekday", "Order_Month", "Ship Mode", "Segment", "Region", "Category"]
X = pd.get_dummies(order_df[features], drop_first=True)
y_ab = order_df['Sales_binary_ab']
y_gb = order_df['Sales_binary_gb']

# Drop NaNs
mask = ~y_gb.isnull() & ~X.isnull().any(axis=1)
X = X[mask].reset_index(drop=True)
y_ab = y_ab[mask].reset_index(drop=True)
y_gb = y_gb[mask].reset_index(drop=True)

print("Final shape:", X.shape)
print("Classification Target Distribution ada boost:\n", y_ab.value_counts())


Final shape: (4916, 33)
Classification Target Distribution ada boost:
 Sales_binary_ab
-1.0    3143
 1.0    1773
Name: count, dtype: int64


We will create encode the categorical features numerically that is new binary target from `Sales_bin` and one-hot encode the other features.

In [ ]:
df.head()

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


In [ ]:
# Split training and test
X_train, X_test, y_train_ab, y_test_ab = train_test_split(X, y_ab, test_size=0.3, random_state=42)
_, _, y_train_gb, y_test_gb = train_test_split(X, y_gb, test_size=0.3, random_state=42)


# 3 Binary Adaboost

In [ ]:
# Create dataframe
def createDF(X, y):
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)
    assert X.shape[0] == y.shape[0], 'Sorry, cannot create dataframe: shape of X and y are not the same.'
    df = DataFrame(X).assign(Class = y)
    return df.dropna().reset_index(drop=True)

# Compute error rate, alpha and w
def compute_error(y, y_pred, w_i):
    '''
    Calculate the error rate of a weak classifier m. Arguments:
    y: actual target value
    y_pred: predicted value by weak classifier
    w_i: individual weights for each observation

    Note that all arrays should be the same length
    '''
    return (sum(w_i * (np.not_equal(y, y_pred)).astype(int)))/sum(w_i)

def compute_alpha(error, learningRate=1):
    '''
    Calculate the weight of a weak classifier m in the majority vote of the final classifier. This is called
    alpha in chapter 10.1 of The Elements of Statistical Learning. Arguments:
    error: error rate from weak classifier m
    '''
    return learningRate * np.log((1 - error) / error)

def update_weights(w_i, alpha, y, y_pred):
    '''
    Update individual weights w_i after a boosting iteration. Arguments:
    w_i: individual weights for each observation
    y: actual target value
    y_pred: predicted value by weak classifier
    alpha: weight of weak classifier used to estimate y_pred
    '''
    return w_i * np.exp(alpha * (np.not_equal(y, y_pred)).astype(int))

class AdaBoost:

    def __init__(self):
        self.alphas = []
        self.models = []
        self.M = None
        self.training_errors = []
        self.prediction_errors = []

    def fit(self, X, y, M = 100):
        '''
        Fit model. Arguments:
        X: independent variables - array-like matrix
        y: target variable - array-like vector
        M: number of boosting rounds. Default is 100 - integer
        '''

        # Clear before calling
        alpha_m, y_pred = 0,0
        self.alphas = []
        self.training_errors = []
        self.M = M

        # Iterate over M weak classifiers
        for m in range(0, M):

            # Set weights for current boosting iteration
            if m == 0:
                w_i = np.ones(len(y)) * 1 / len(y)  # At m = 0, weights are all the same and equal to 1 / N
            else:
                # (d) Update w_i
                w_i = update_weights(w_i, alpha_m, y, y_pred)
                w_i /= np.sum(w_i)

            # (a) Fit weak classifier and predict labels
            model = DecisionTreeClassifier(max_depth = 1)     # Stump: Two terminal-node classification tree
            model.fit(X, y, sample_weight = w_i)
            y_pred = model.predict(X)

            self.models.append(model) # Save to list of weak classifiers

            # (b) Compute error
            error_m = compute_error(y, y_pred, w_i)
            self.training_errors.append(error_m)

            # (c) Compute alpha
            alpha_m = compute_alpha(error_m)
            self.alphas.append(alpha_m)

        assert len(self.models) == len(self.alphas)


    def predict(self, X):
        '''
        Predict using fitted model. Arguments:
        X: independent variables - array-like
        '''

        # Initialise dataframe with weak predictions for each observation
        weak_preds = pd.DataFrame(index = range(len(X)), columns = range(self.M))

        # Predict class label for each weak classifier, weighted by alpha_m
        for m in range(self.M):
            y_pred_m = self.models[m].predict(X) * self.alphas[m]
            weak_preds[m] = y_pred_m

        # Calculate final predictions
        y_pred = (1 * np.sign(weak_preds.T.sum())).astype(int)

        return y_pred

def get_metrics(y_true, y_pred):

    precision = precision_score(y_true, y_pred, average='weighted')
    recall= recall_score(y_true, y_pred, average='weighted')
    f1= f1_score(y_true, y_pred, average='weighted')
    accuracy = accuracy_score(y_true, y_pred)

    return {"precision": precision, "recall": recall, 'f1_score': f1, 'accuracy': accuracy }


## 3.1 Evaluation metrics for Binary Adaboost

In [ ]:
# Train & Evaluate AdaBoost
model = AdaBoost()
model.fit(X_train, y_train_ab, M=50)
y_pred_ab = model.predict(X_test)

print("AdaBoost Model Metrics:")
print(f"Accuracy : {accuracy_score(y_test_ab, y_pred_ab):.4f}")
print(f"Precision: {precision_score(y_test_ab, y_pred_ab):.4f}")
print(f"Recall   : {recall_score(y_test_ab, y_pred_ab):.4f}")
print(f"F1-Score : {f1_score(y_test_ab, y_pred_ab):.4f}")

print("\n[AdaBoost Classification Report]")
print(classification_report(y_test_ab, y_pred_ab, target_names=['Low Sales (-1)', 'High Sales (1)']))

AdaBoost Model Metrics:
Accuracy : 0.7397
Precision: 0.6612
Recall   : 0.5421
F1-Score : 0.5958

[AdaBoost Classification Report]
                precision    recall  f1-score   support

Low Sales (-1)       0.77      0.85      0.81       953
High Sales (1)       0.66      0.54      0.60       522

      accuracy                           0.74      1475
     macro avg       0.72      0.69      0.70      1475
  weighted avg       0.73      0.74      0.73      1475



# 4 Gradient Boost

In [ ]:
import pandas as pd, numpy as np
from pandas import DataFrame, Series
from sklearn.tree import DecisionTreeRegressor # Added import

def createDF(X, y):
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)
    assert X.shape[0] == y.shape[0], 'Sorry, cannot create dataframe: shape of X and y are not the same.'
    df = DataFrame(X).assign(Class = y)
    return df.dropna().reset_index(drop=True)

class GradientBoost:
    def __init__(self, learning_rate=0.1, num_iter=1000):
        self.learning_rate = learning_rate
        self.num_iter = num_iter
        self.y_mean = 0

    def compute_loss(self, y, y_pred):
        return  0.5 * np.sum(np.square(y - y_pred))


    def compute_residuals(self, y, y_pred):
        return  y-y_pred


    def create_model(self, X, y):
        model = DecisionTreeRegressor(max_depth=1, criterion='squared_error')
        model.fit(X,y)
        return model

    def predict(self, models, X):
        # Use self.y_mean calculated during training
        initial_pred = np.array([self.y_mean] * len(X))
        pred = initial_pred # Keep pred as 1D

        for i in range(len(models)):
            new_prediction = models[i].predict(X) # Keep prediction as 1D
            pred += self.learning_rate * new_prediction

        return pred

    def train(self, X, y):
        models = []
        losses = []

        y_np = y.values.flatten() # Convert y Series to 1D numpy array

        self.y_mean = np.mean(y_np)
        initial_pred = np.array([self.y_mean] * len(y_np))
        pred = initial_pred # Keep pred as 1D

        for epoch in range(self.num_iter):
            loss      = self.compute_loss(y_np, pred) # Use y_np
            residuals = self.compute_residuals(y_np, pred) # Use y_np
            model     = self.create_model(X, residuals)


            new_model_pred = model.predict(X) # Keep prediction as 1D
            pred = pred + self.learning_rate * new_model_pred

            losses.append(loss)
            models.append(model)


        return models, losses

## 4.1 Evaluation metric of Gradient boost

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Initialize and train the GradientBoostingClassifier
gbc_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=1, random_state=42)
gbc_model.fit(X_train, y_train_gb)

# Make predictions
y_pred_gbc = gbc_model.predict(X_test)

# Evaluate the model
metrics_gbc = get_metrics(y_test_gb, y_pred_gbc)

print("Gradient Boosting Model Metrics:")
print(f"Accuracy : {accuracy_score(y_test_gb, y_pred_gbc):.4f}")
print(f"Precision: {precision_score(y_test_gb, y_pred_gbc):.4f}")
print(f"Recall   : {recall_score(y_test_gb, y_pred_gbc):.4f}")
print(f"F1-Score : {f1_score(y_test_gb, y_pred_gbc):.4f}\n")

print(classification_report(y_test_gb, y_pred_gbc, target_names=['Low Sales (0)', 'High Sales (1)']))


Gradient Boosting Model Metrics:
Accuracy : 0.7410
Precision: 0.6434
Recall   : 0.6015
F1-Score : 0.6218

                precision    recall  f1-score   support

 Low Sales (0)       0.79      0.82      0.80       953
High Sales (1)       0.64      0.60      0.62       522

      accuracy                           0.74      1475
     macro avg       0.72      0.71      0.71      1475
  weighted avg       0.74      0.74      0.74      1475



## 4.2 Mean square error

In [ ]:
from sklearn.metrics import mean_squared_error # Added import
from sklearn.ensemble import GradientBoostingRegressor # Moved here for clarity

for lr in [0.1, .4, .5, .8]:
    G = GradientBoost(learning_rate=lr)
    models, losses = G.train(X_train, y_train_gb)
    prediction = G.predict(models, X_test) # Updated predict call

    print(f'RMSE from Manual Implementation for lr: {lr}', 1 - np.sqrt(mean_squared_error(y_test_gb.values, prediction))) # Use y_test.values


model = GradientBoostingRegressor(n_estimators=1000, max_depth=1, criterion='squared_error')

model.fit(X_train, y_train_gb.values) # Use y_train.values
y_pred_sklearn = model.predict(X_test) # Renamed to avoid conflict
print('RMSE from Sklearn Implementation:', 1 - np.sqrt(mean_squared_error(y_test_gb.values, y_pred_sklearn))) # Use y_test.values

RMSE from Manual Implementation for lr: 0.1 0.584206694247586
RMSE from Manual Implementation for lr: 0.4 0.5835225893750123
RMSE from Manual Implementation for lr: 0.5 0.5835021278756141
RMSE from Manual Implementation for lr: 0.8 0.5834756362971807
RMSE from Sklearn Implementation: 0.584206694247586


# 5 Multi Class AdaBoost

In [ ]:
y_multiclass = order_df.loc[X.index, 'Sales_bin']

# Ensure the target is of string type for consistent handling by metrics function
y_multiclass = y_multiclass.astype(str)

# Split the data into training and testing sets for multi-class classification
X_train_mc, X_test_mc, y_train_mc, y_test_mc = train_test_split(
    X, y_multiclass, test_size=0.3, random_state=42
)

# Initialize the base estimator (Decision Tree Classifier)
base_estimator_mc = DecisionTreeClassifier(max_depth=3, random_state=42)

# Initialize and train the AdaBoost Classifier for multi-class problems
adaboost_multiclass_model = AdaBoostClassifier(
    estimator=base_estimator_mc,
    n_estimators=100,
    learning_rate=0.1,
    algorithm='SAMME',
    random_state=42
)

adaboost_multiclass_model.fit(X_train_mc, y_train_mc)

# Make predictions on the test set
y_pred_multiclass = adaboost_multiclass_model.predict(X_test_mc)

# Evaluate the multi-class AdaBoost model using the predefined get_metrics function
metrics_multiclass_adaboost = get_metrics(y_test_mc, y_pred_multiclass)

metrics_multiclass_adaboost

{'precision': 0.26150357658832235,
 'recall': 0.42847457627118646,
 'f1_score': 0.2681759738982102,
 'accuracy': 0.42847457627118646}